# IT25102957 - Decision Tree Classifier

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25102957 - Decision Tree Classifier

**Theoretical Justification & Model Suitability:**
- **Suitability:** Credit card default decisions follow hierarchical, conditional logic (e.g., *if customer delayed payment for >2 months and credit utilization is >80%, default probability surges*). Decision Trees capture these non-linear thresholds without assuming monotonic or linear distributions.
- **Tuning Strategy:** Evaluates multiple tree depths (`max_depth`: 3, 5, 8, 12, None), split criteria (`gini` vs `entropy`), and leaf constraints (`min_samples_leaf`) using 3-fold cross-validation scored on F1.
- **Generalization Control:** Pruning controls variance; unconstrained trees suffer severe memorization/overfitting, whereas constrained depth achieves optimal generalization.


In [ ]:
# ==============================================================================
# IT25102957 — Decision Tree Classifier (Systematic Hyperparameter Tuning & Pruning)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (ConfusionMatrixDisplay, f1_score,
                             roc_auc_score, RocCurveDisplay)

# ------------------------------------------------------------------------------
# 1. GridSearchCV over depth, criterion, leaf size + class_weight (for imbalance)
# ------------------------------------------------------------------------------
dt_params = {
    'max_depth':        [3, 5, 8, 12, None],
    'criterion':        ['gini', 'entropy'],
    'min_samples_leaf': [1, 5, 10],
    'class_weight':     [None, 'balanced']          # handles 22% default rate
}
dt_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_params, scoring='f1', cv=3, n_jobs=-1
)
dt_search.fit(X_train, y_train)
best_dt = dt_search.best_estimator_

print('=== IT25102957: Decision Tree Optimization ===')
print(f'Optimal Parameters: {dt_search.best_params_}')
print(f'Training Cross-Validation F1: {dt_search.best_score_:.4f}')

# ------------------------------------------------------------------------------
# 2. Print Tree Depth and Feature Importances
# ------------------------------------------------------------------------------
print(f"\nTrained tree depth         : {best_dt.get_depth()}")
print(f"Trained tree leaves        : {best_dt.get_n_leaves()}")

feat_imp = pd.Series(best_dt.feature_importances_,
                     index=X_train.columns).sort_values(ascending=False)
print("\nFeature Importances (top 10):")
print(feat_imp.head(10).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
top = feat_imp.head(10).iloc[::-1]
ax.barh(top.index, top.values, color='#2980b9', edgecolor='#1a5276')
ax.set_title('IT25102957 Decision Tree: Top 10 Feature Importances',
             fontweight='bold')
ax.set_xlabel('Gini Importance')
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 3. Visualize the ACTUAL best_dt (truncated to depth 3)
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(best_dt,
          feature_names=list(X_train.columns),
          class_names=['No Default', 'Default'],
          filled=True, rounded=True, fontsize=8,
          max_depth=3,                    # <-- truncates RENDER, not the model
          ax=ax)
ax.set_title(f'IT25102957: Best Decision Tree Structure '
             f'(displayed to depth 3 of {best_dt.get_depth()})',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 4. max_depth sweep (1, 3, 5, 10, None) for variety
# ------------------------------------------------------------------------------
print("\n=== max_depth Sweep  ===")
depth_rows = []
for d in [1, 3, 5, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1]
    depth_rows.append({
        'max_depth': d,
        'Depth':     m.get_depth(),
        'Leaves':    m.get_n_leaves(),
        'Accuracy':  (y_pred == y_test).mean(),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_prob)
    })
depth_df = pd.DataFrame(depth_rows)
display(depth_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(depth_df)), depth_df['F1'], 'o-',
        label='F1', color='#c0392b', linewidth=2)
ax.plot(range(len(depth_df)), depth_df['ROC-AUC'], 's-',
        label='ROC-AUC', color='#2980b9', linewidth=2)
ax.set_xticks(range(len(depth_df)))
ax.set_xticklabels([str(d) for d in depth_df['max_depth']])
ax.set_xlabel('max_depth (None = unconstrained)')
ax.set_ylabel('Score')
ax.set_title('IT25102957 Decision Tree: Overfitting Curve vs max_depth',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 5. Inline metrics (Summary Metrics)
# ------------------------------------------------------------------------------
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, classification_report)
y_pred_dt = best_dt.predict(X_test)
print("\n=== Best Decision Tree: Test-Set Metrics ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, best_dt.predict_proba(X_test)[:,1]):.4f}")
print("\n" + classification_report(y_test, y_pred_dt,
                                    target_names=['No Default','Default']))

# ------------------------------------------------------------------------------
# 6. Confusion Matrix + ROC Curve
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    best_dt, X_test, y_test, cmap='Blues',
    display_labels=['No Default', 'Default'], ax=ax)
ax.set_title('IT25102957 Decision Tree: Test Confusion Matrix',
             fontweight='bold')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_estimator(best_dt, X_test, y_test, ax=ax,
                               name='Decision Tree')
ax.plot([0,1],[0,1],'--',color='gray',label='Chance')
ax.legend(); ax.set_title('IT25102957 Decision Tree: ROC Curve',
                          fontweight='bold')
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------------
# 7. Register
# ------------------------------------------------------------------------------
record_model('IT25102957', 'Decision Tree Classifier', best_dt)
